# Pilot scaling test: computing κ_φ on a one-megapixel field

**Purpose.** Section D of the statement promises *"matrix-free sensitivity solves ... will scale from one-megapixel fields to full multi-sensor exposures."* This notebook tests that promise: can the stability measure κ_φ be computed on a 1024 × 1024 detector, and at what cost in time and memory?

**Why "matrix-free" is necessary.** At one megapixel with four dithered exposures, there are about 4 million measurements and about 2,500 stars. The full Jacobian $J$ would be a dense matrix of roughly **4 million × 2,500 entries, about 85 GB**. It cannot be stored on a laptop. Instead, we only ever compute **products** with $J$ and $J^\top$, by rendering the sky, cropping it to each exposure, and multiplying by the gain. These cost about as much as simulating the data once.

**How κ_φ is computed at scale.** The Fisher information has two blocks: image ($a$) and gain ($\theta$). Using the same Schur-complement split as Section B of the statement:

$$\kappa_\varphi^2 = \underbrace{e_j^\top F_{aa}^{-1} e_j}_{\text{known-gain term}} + \underbrace{z_j^\top S^{-1} z_j}_{\text{calibration term}}, \qquad Z = F_{aa}^{-1}F_{a\theta}, \quad S = F_{\theta\theta} - F_{\theta a} Z,$$

where $z_j$ is row $j$ of $Z$. Two consequences:
1. **The calibration term for every star costs only $m$ solves**, where $m$ is the number of gain modes (here 9), however many stars there are.
2. The known-gain term needs **one solve per target star**.

Each solve uses the **conjugate gradient** method with a diagonal (Jacobi) preconditioner, and never forms $F_{aa}$.

**Model.** Same as the Poisson pilot: Poisson weights $1/\mu$, four dithered exposures of 2 pixels, smooth extended emission, and stars on a jittered grid (about one star per 20 × 20 pixels, so star density stays constant as the field grows).

**Sections**
1. Setup and matrix-free operators
2. Correctness check against a dense calculation (small field)
3. Scaling from 128 × 128 to 1024 × 1024
4. Figures: runtime and memory
5. Writing the result, limitations, interview questions

**Requirements:** `numpy`, `pandas`, `matplotlib`. Runtime is about 1 to 2 minutes on a laptop; the 1024 × 1024 field takes most of it.

## 1. Setup

In [ ]:
import time, tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ---- configuration ----------------------------------------------------------
MAX_SHIFT = 4
PSF_SIGMA = 1.2
STAMP_R = 5              # star stamp half-width in pixels (about 4 PSF widths)
SKY = 100.0              # expected sky counts per pixel
EXT_AMP = 10.0
FLUX_RANGE = (500.0, 5000.0)
GAIN_RMS = 0.03
EXT_ORDER = 3            # 9 extended-emission modes
GAIN_ORDER = 3           # 9 gain modes (m = 9)
STAR_SPACING = 20        # one star per ~20 x 20 pixels
JITTER = 6
DITHER_AMP = 2

SIZES = [128, 256, 512, 1024]   # detector side lengths to test
N_TARGETS = 10                  # stars that get a full kappa (known-gain term) at each size
RUN_4MP = False                 # set True to also try 2048 x 2048 (about 4x the 1024 cost)
CG_TOL = 1e-10

BLUE, ORANGE = "#2a78d6", "#eb6834"

### Matrix-free operators
- `render(a)`: builds the sky image from the parameters (sky level + smooth modes + star stamps). `render_adj` is its exact adjoint (transpose).
- `crop` / `pad_add`: the dither window $A_k$ and its adjoint.
- `F_aa(v)`, `F_a_theta(φ)`, `F_theta_theta()`: blocks of the Poisson Fisher information, applied without ever forming $J$.
- `diag_F_aa()`: the exact diagonal of $F_{aa}$, used as the conjugate-gradient preconditioner.
- `kappa_matrix_free`: the Schur-complement route described above.

In [ ]:
def cos_vectors(size, max_order):
    idx = (np.arange(size) + 0.5) / size
    return np.array([np.cos(np.pi * u * idx) for u in range(max_order + 1)])  # (U, size)


def mode_list(max_order):
    return [(u, v) for u in range(max_order + 1) for v in range(max_order + 1) if 1 <= u + v <= max_order]


class MatrixFreeProblem:
    """Joint image/gain model with Poisson weights, all operators applied without forming J."""

    def __init__(self, n_det, rng, amp=DITHER_AMP):
        self.N = n_det
        self.S = n_det + MAX_SHIFT
        S = self.S
        # star positions on a jittered grid inside the region every window sees
        lo, hi = MAX_SHIFT + STAMP_R + 1, n_det - STAMP_R - 2
        grid = np.arange(lo + JITTER, hi - JITTER + 1, STAR_SPACING)
        gy, gx = np.meshgrid(grid, grid, indexing="ij")
        pos = np.stack([gy.ravel(), gx.ravel()], 1).astype(float)
        pos += rng.uniform(-JITTER, JITTER, size=pos.shape)
        self.pos = pos
        self.n_stars = len(pos)
        # stamps: (n_stars, (2r+1)^2) values and flat scene indices
        r = STAMP_R
        off = np.arange(-r, r + 1)
        cy = np.round(pos[:, 0]).astype(int); cx = np.round(pos[:, 1]).astype(int)
        yy = cy[:, None, None] + off[None, :, None]
        xx = cx[:, None, None] + off[None, None, :]
        val = np.exp(-((yy - pos[:, 0, None, None]) ** 2 + (xx - pos[:, 1, None, None]) ** 2) / (2 * PSF_SIGMA ** 2))
        val /= val.sum(axis=(1, 2), keepdims=True)
        self.st_idx = (yy * S + xx).reshape(self.n_stars, -1)
        self.st_val = val.reshape(self.n_stars, -1)
        # extended-emission modes on the scene grid (separable cosines, unit rms)
        self.ext_modes = mode_list(EXT_ORDER)
        self.Cs = cos_vectors(S, EXT_ORDER)
        self.ext_norm = np.array([np.sqrt(np.mean(self.Cs[u] ** 2) * np.mean(self.Cs[v] ** 2)) for u, v in self.ext_modes])
        self.n_ext = len(self.ext_modes)
        # gain modes on the detector grid
        self.gain_modes = mode_list(GAIN_ORDER)
        self.Cd = cos_vectors(n_det, GAIN_ORDER)
        self.gain_norm = np.array([np.sqrt(np.mean(self.Cd[u] ** 2) * np.mean(self.Cd[v] ** 2)) for u, v in self.gain_modes])
        self.m = len(self.gain_modes)
        self.p = 1 + self.n_ext + self.n_stars
        self.offs = [(0, 0), (amp, 0), (0, amp), (amp, amp)]
        # true parameters (redraw extended emission if the sky would dip too low)
        while True:
            ext_c = EXT_AMP * rng.standard_normal(self.n_ext)
            if self.render_ext(ext_c).min() > 0.2 * SKY - SKY:
                break
        self.a = np.concatenate([[SKY], ext_c, rng.uniform(*FLUX_RANGE, self.n_stars)])
        self.theta = GAIN_RMS * rng.standard_normal(self.m)
        # fixed quantities at the true parameters
        self.x = self.render(self.a)
        self.g = self.gain(self.theta)
        self.xk = [self.crop(self.x, o) for o in self.offs]
        self.W = [1.0 / (self.g * xk) for xk in self.xk]      # Poisson weights 1/mu

    # ---- scene rendering and its adjoint -----------------------------------
    def render_ext(self, c):
        C = np.zeros((EXT_ORDER + 1, EXT_ORDER + 1))
        for (u, v), cc, nn in zip(self.ext_modes, c, self.ext_norm):
            C[u, v] = cc / nn
        return self.Cs.T @ C @ self.Cs

    def render(self, a):
        S = self.S
        img = np.full(S * S, a[0])
        img += self.render_ext(a[1:1 + self.n_ext]).ravel()
        np.add.at(img, self.st_idx.ravel(), (a[1 + self.n_ext:, None] * self.st_val).ravel())
        return img.reshape(S, S)

    def render_adj(self, Y):
        out = np.empty(self.p)
        out[0] = Y.sum()
        G = self.Cs @ Y @ self.Cs.T
        out[1:1 + self.n_ext] = [G[u, v] / nn for (u, v), nn in zip(self.ext_modes, self.ext_norm)]
        out[1 + self.n_ext:] = (Y.ravel()[self.st_idx] * self.st_val).sum(1)
        return out

    def crop(self, X, o):
        dy, dx = o
        return X[dy:dy + self.N, dx:dx + self.N]

    def pad_add(self, acc, R, o):
        dy, dx = o
        acc[dy:dy + self.N, dx:dx + self.N] += R

    # ---- gain and its adjoint ------------------------------------------------
    def gain(self, th):
        C = np.zeros((GAIN_ORDER + 1, GAIN_ORDER + 1))
        for (u, v), t, nn in zip(self.gain_modes, th, self.gain_norm):
            C[u, v] = t / nn
        return 1.0 + self.Cd.T @ C @ self.Cd

    def gain_adj(self, R):
        G = self.Cd @ R @ self.Cd.T
        return np.array([G[u, v] / nn for (u, v), nn in zip(self.gain_modes, self.gain_norm)])

    # ---- Fisher-information blocks (matrix-free) ------------------------------
    def F_aa(self, v):
        X = self.render(v)
        acc = np.zeros((self.S, self.S))
        for o, W in zip(self.offs, self.W):
            self.pad_add(acc, W * self.g ** 2 * self.crop(X, o), o)
        return self.render_adj(acc)

    def F_a_theta(self, phi):
        Bphi = self.gain(phi) - 1.0
        acc = np.zeros((self.S, self.S))
        for o, W, xk in zip(self.offs, self.W, self.xk):
            self.pad_add(acc, W * self.g * xk * Bphi, o)
        return self.render_adj(acc)

    def F_theta_theta(self):
        D = sum(W * xk ** 2 for W, xk in zip(self.W, self.xk))
        F = np.zeros((self.m, self.m))
        for n in range(self.m):
            e = np.zeros(self.m); e[n] = 1.0
            F[:, n] = self.gain_adj(D * (self.gain(e) - 1.0))
        return F

    def diag_F_aa(self):
        acc = np.zeros((self.S, self.S))
        for o, W in zip(self.offs, self.W):
            self.pad_add(acc, W * self.g ** 2, o)
        d = np.empty(self.p)
        d[0] = acc.sum()
        Cs2 = self.Cs ** 2
        G = Cs2 @ acc @ Cs2.T
        d[1:1 + self.n_ext] = [G[u, v] / nn ** 2 for (u, v), nn in zip(self.ext_modes, self.ext_norm)]
        d[1 + self.n_ext:] = (acc.ravel()[self.st_idx] * self.st_val ** 2).sum(1)
        return d


def pcg(matvec, b, Minv, tol=1e-10, maxiter=5000):
    x = np.zeros_like(b); r = b.copy(); z = Minv * r; p = z.copy(); rz = r @ z
    bnorm = np.linalg.norm(b)
    for it in range(1, maxiter + 1):
        Ap = matvec(p); alpha = rz / (p @ Ap)
        x += alpha * p; r -= alpha * Ap
        if np.linalg.norm(r) < tol * bnorm:
            return x, it
        z = Minv * r; rz_new = r @ z; p = z + (rz_new / rz) * p; rz = rz_new
    return x, maxiter


def kappa_matrix_free(prob, target_stars, tol=1e-10):
    """Schur-complement route: m solves give the calibration term for EVERY star;
    one more solve per target gives its known-gain term."""
    Minv = 1.0 / prob.diag_F_aa()
    iters = []
    Z = np.zeros((prob.p, prob.m))
    for n in range(prob.m):
        e = np.zeros(prob.m); e[n] = 1.0
        Z[:, n], it = pcg(prob.F_aa, prob.F_a_theta(e), Minv, tol); iters.append(it)
    FtaZ = np.array([prob.gain_adj(sum(W * prob.g * xk * prob.crop(prob.render(Z[:, n]), o)
                                        for o, W, xk in zip(prob.offs, prob.W, prob.xk))) for n in range(prob.m)]).T
    S = prob.F_theta_theta() - FtaZ
    S = 0.5 * (S + S.T)
    star0 = 1 + prob.n_ext
    Zs = Z[star0:, :]
    calib_all = np.einsum("ij,ij->i", Zs @ np.linalg.inv(S), Zs)   # calibration term, every star
    known, joint = [], []
    for j in target_stars:
        e = np.zeros(prob.p); e[star0 + j] = 1.0
        u, it = pcg(prob.F_aa, e, Minv, tol); iters.append(it)
        known.append(u[star0 + j]); joint.append(u[star0 + j] + calib_all[j])
    return np.array(known), np.array(joint), calib_all, iters

## 2. Correctness check against a dense calculation

On a small 64 × 64 field, the full Jacobian is small enough to build explicitly. We compute κ_φ both ways. **The relative error should be tiny (about 1e-8 or smaller)**; otherwise the matrix-free operators contain a bug and the scaling numbers below cannot be trusted.

In [ ]:
def dense_kappa(prob):
    p, m = prob.p, prob.m
    cols_a = []
    for i in range(p):
        e = np.zeros(p); e[i] = 1.0; X = prob.render(e)
        cols_a.append(np.concatenate([(np.sqrt(W) * prob.g * prob.crop(X, o)).ravel()
                                      for o, W in zip(prob.offs, prob.W)]))
    cols_t = []
    for n in range(m):
        e = np.zeros(m); e[n] = 1.0; Bp = prob.gain(e) - 1.0
        cols_t.append(np.concatenate([(np.sqrt(W) * xk * Bp).ravel() for W, xk in zip(prob.W, prob.xk)]))
    Ja, Jt = np.array(cols_a).T, np.array(cols_t).T
    J = np.hstack([Ja, Jt])
    s0 = 1 + prob.n_ext
    Finv, Fa_inv = np.linalg.inv(J.T @ J), np.linalg.inv(Ja.T @ Ja)
    idx = s0 + np.arange(prob.n_stars)
    return Fa_inv[idx, idx], Finv[idx, idx]

check = MatrixFreeProblem(64, np.random.default_rng(0))
k_dense, j_dense = dense_kappa(check)
k_mf, j_mf, _, _ = kappa_matrix_free(check, np.arange(check.n_stars), tol=CG_TOL)
print(f"64 x 64 field: {check.n_stars} stars, {check.p} image parameters, {check.m} gain modes")
print(f"max relative error, known-gain kappa^2: {np.max(np.abs(k_mf - k_dense) / k_dense):.1e}")
print(f"max relative error, joint kappa^2:      {np.max(np.abs(j_mf - j_dense) / j_dense):.1e}")

## 3. Scaling from 128 × 128 to 1024 × 1024

For each detector size we record:

| Column | Meaning |
|---|---|
| `pixels` | Detector pixels per exposure |
| `measurements` | Pixels × 4 exposures |
| `stars`, `image params` | Number of stars and of image parameters |
| `dense J (GB)` | Memory a dense Jacobian **would** need (not allocated) |
| `peak memory (MB)` | Actual peak memory of the matrix-free computation (Python's `tracemalloc`) |
| `calibration term, all stars (s)` | Time for the $m$ gain solves plus $S$: gives the calibration term for **every** star |
| `per target star (s)` | Average time for one known-gain solve |
| `mean CG iterations` | Iterations per solve; **roughly constant means the method scales well** |
| `star inflation (median)` | Joint ÷ known-gain variance for the target stars |

In [ ]:
def run_size(n_det, seed=0):
    tracemalloc.start()
    prob = MatrixFreeProblem(n_det, np.random.default_rng(seed))
    Minv = 1.0 / prob.diag_F_aa()
    t0 = time.perf_counter()
    iters = []
    Z = np.zeros((prob.p, prob.m))
    for n in range(prob.m):
        e = np.zeros(prob.m); e[n] = 1.0
        Z[:, n], it = pcg(prob.F_aa, prob.F_a_theta(e), Minv, CG_TOL); iters.append(it)
    FtaZ = np.array([prob.gain_adj(sum(W * prob.g * xk * prob.crop(prob.render(Z[:, n]), o)
                                        for o, W, xk in zip(prob.offs, prob.W, prob.xk)))
                     for n in range(prob.m)]).T
    S = prob.F_theta_theta() - FtaZ; S = 0.5 * (S + S.T)
    s0 = 1 + prob.n_ext
    calib_all = np.einsum("ij,ij->i", Z[s0:] @ np.linalg.inv(S), Z[s0:])
    t1 = time.perf_counter()
    targets = np.linspace(0, prob.n_stars - 1, min(N_TARGETS, prob.n_stars)).astype(int)
    known = []
    for j in targets:
        e = np.zeros(prob.p); e[s0 + j] = 1.0
        u, it = pcg(prob.F_aa, e, Minv, CG_TOL); iters.append(it)
        known.append(u[s0 + j])
    t2 = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    known = np.array(known)
    return {"detector": f"{n_det} x {n_det}", "pixels": n_det ** 2, "measurements": 4 * n_det ** 2,
            "stars": prob.n_stars, "image params": prob.p,
            "dense J (GB)": 4 * n_det ** 2 * (prob.p + prob.m) * 8 / 1e9,
            "peak memory (MB)": peak / 1e6,
            "calibration term, all stars (s)": t1 - t0,
            "per target star (s)": (t2 - t1) / len(targets),
            "mean CG iterations": np.mean(iters),
            "star inflation (median)": np.median((known + calib_all[targets]) / known)}

rows = []
for n in SIZES + ([2048] if RUN_4MP else []):
    rows.append(run_size(n))
    print(f"done: {n} x {n}")
scaling = pd.DataFrame(rows)
display(scaling.round({"dense J (GB)": 2, "peak memory (MB)": 1, "calibration term, all stars (s)": 2,
                       "per target star (s)": 3, "mean CG iterations": 1, "star inflation (median)": 2}))

**How to read this table**
- `peak memory` should be **orders of magnitude smaller** than `dense J`. That is the point of the matrix-free method.
- `mean CG iterations` should stay roughly constant as the field grows. Then the cost per solve grows only with the number of pixels, i.e. linearly.
- `star inflation (median)` may **fall** as the field grows. More stars observe the same 9 gain modes, so the gain is pinned down better and each star pays less. That is a scientific result in its own right, and worth thinking about before an interview.
- Runtime depends on your computer. Report your own numbers and say what machine you used.

## 4. Figures
Two separate charts, since time and memory are different quantities.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), dpi=120)
px = scaling["pixels"]
ax = axs[0]
ax.plot(px, scaling["calibration term, all stars (s)"], color=BLUE, lw=2, marker="o", ms=6,
        label="Calibration term, all stars")
ax.plot(px, scaling["per target star (s)"], color=ORANGE, lw=2, marker="o", ms=6, label="Known-gain term, per star")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Detector pixels"); ax.set_ylabel("Runtime (seconds)")
ax.set_title("Runtime", fontsize=10)
ax = axs[1]
ax.plot(px, scaling["peak memory (MB)"] / 1e3, color=BLUE, lw=2, marker="o", ms=6, label="Matrix-free (measured)")
ax.plot(px, scaling["dense J (GB)"], color=ORANGE, lw=2, marker="o", ms=6, ls="--", label="Dense Jacobian (would need)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Detector pixels"); ax.set_ylabel("Memory (GB)")
ax.set_title("Memory", fontsize=10)
for ax in axs:
    ax.set_xticks(px); ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v ** 0.5))}\u00b2"))
    ax.grid(True, color="#e6e6e6", lw=0.8)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); fig.savefig("scaling_runtime_memory.png", dpi=200); plt.show()

## 5. Writing the result

Use **your own numbers** and say what machine you used. A possible sentence for Section D of the statement:

> In a pilot on a 1024 × 1024 field with about **[N]** stars, matrix-free solves computed κ_φ in **[T]** seconds using **[M]** MB of memory, versus **[G]** GB for an explicit Jacobian, with conjugate-gradient iterations nearly constant across field sizes.

A shorter version, if space is tight:

> A matrix-free pilot computed κ_φ for a one-megapixel field in **[T]** s and **[M]** MB on a laptop.

**Be honest.** If your machine is slower, or iterations grow with size, report that. The claim to support is that the method is feasible at survey-relevant sizes, not that it is instantaneous.

### Sanity checklist
1. The dense-versus-matrix-free error in Section 2 is tiny.
2. `mean CG iterations` is roughly constant across sizes.
3. `peak memory` is far below `dense J`.

### Limitations (be ready to state these)
- Runtime and memory are for computing κ_φ at the true parameters, not for the full reconstruction; the reconstruction itself uses the same operators, so its cost per iteration is similar.
- Single-threaded NumPy on a CPU; a GPU would be faster, as Section D proposes.
- One megapixel per exposure; real survey cameras have many sensors, but each sensor tile can be processed in the same way.
- Stars on a jittered grid with known positions; real fields have crowding and blending.

### Questions you may be asked in an interview
- Why does the calibration term for every star cost only $m$ solves? *(Hint: $z_j$ is row $j$ of $Z = F_{aa}^{-1}F_{a\theta}$, which has only $m$ columns.)*
- Why does the number of conjugate-gradient iterations stay nearly constant? *(Hint: the Jacobi-preconditioned $F_{aa}$ is well conditioned because stars barely overlap.)*
- Why does the star penalty shrink as the field grows?
- How would you get κ_φ for **all** stars cheaply, not just a sample? *(Hint: estimate the diagonal of $F_{aa}^{-1}$, for example by probing, or use the near-block-diagonal structure.)*